In [3]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm

In [5]:
train_clean_chunks = np.load('data/train_clean_chunks.npy')
train_noisy_chunks = np.load('data/train_noisy_chunks.npy')

In [6]:
torch_train_noisy_chunks = torch.from_numpy(train_noisy_chunks)
torch_train_clean_chunks = torch.from_numpy(train_clean_chunks)

In [7]:
full_dataset = torch.utils.data.TensorDataset(torch_train_noisy_chunks, torch_train_clean_chunks)

In [9]:
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])

In [11]:
len(train_dataset),  len(val_dataset)

(22464, 5615)

In [13]:
train_dataloader = DataLoader(train_dataset, batch_size=32,)
val_dataloader = DataLoader(val_dataset, batch_size=32,)

In [28]:
class DenoisingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, 3, padding=1)
        self.layer2 = nn.Conv1d(16, 32, 3, padding=1)
        self.layer3 = nn.Conv1d(32, 16, 3, padding=1)
        self.layer4 = nn.Conv1d(16, 1, 3, padding=1)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.layer4(x)
        x = x.squeeze(1)
        return x

In [29]:
model = DenoisingModel()
print(model)

DenoisingModel(
  (relu): LeakyReLU(negative_slope=0.01)
  (layer1): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (layer2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (layer3): Conv1d(32, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (layer4): Conv1d(16, 1, kernel_size=(3,), stride=(1,), padding=(1,))
)


In [30]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001 )

In [31]:
n_epochs = 4

for epoch in tqdm(range(n_epochs)):
    epoch_loss = 0
    for batch in train_dataloader:
        noisy_batch, clean_batch = batch
        optimizer.zero_grad()
        train_pred_clean = model(noisy_batch)
        train_loss = loss_fn(train_pred_clean, clean_batch)
        train_loss.backward()
        optimizer.step()
        epoch_loss += train_loss.item()
    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch}, Avg Loss: {avg_epoch_loss:.4f}")
    
    model.eval()
    with torch.no_grad():
        val_epoch_loss = 0
        for batch in val_dataloader:
            noisy_val_batch, clean_val_batch = batch
            val_pred_clean = model(noisy_val_batch)
            val_loss = loss_fn(val_pred_clean, clean_val_batch)
            val_epoch_loss += val_loss.item()
        avg_val_loss = val_epoch_loss / len(val_dataloader)
        print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
    model.train()    

  0%|                                                     | 0/4 [00:00<?, ?it/s]

Epoch 0, Avg Loss: 0.0015


 25%|███████████                                 | 1/4 [14:19<42:58, 859.35s/it]

Epoch 0, Avg Val Loss: 0.0012
Epoch 1, Avg Loss: 0.0012


 50%|██████████████████████                      | 2/4 [28:32<28:31, 855.95s/it]

Epoch 1, Avg Val Loss: 0.0011
Epoch 2, Avg Loss: 0.0011


 75%|█████████████████████████████████           | 3/4 [42:52<14:17, 857.54s/it]

Epoch 2, Avg Val Loss: 0.0011
Epoch 3, Avg Loss: 0.0011


100%|████████████████████████████████████████████| 4/4 [57:08<00:00, 857.09s/it]

Epoch 3, Avg Val Loss: 0.0011


In [ ]:
torch.save(model.state_dict(), 'denoising_cnn_4layers_4epochs.pth')